# Gemma 4 E4B + LoRA: русско-тувинский перевод (Colab, A100)

Обучение на **всём train** из `prepare_data.py` — тот же сплит, что у NLLB v3,
тот же **чистый тест (1369 пар)**, те же промпты `ru→tyv:` / `tyv→ru:`.
Результат сравнивается с эталоном NLLB v3: **49.2 / 49.1** chrF++.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → **A100 GPU**, включить **High-RAM**.

Порядок:
1. Ячейки 1–4 — установка, Диск, Hugging Face, данные.
2. Ячейка 5 оставь `SMOKE_TEST = True` → прогони 5–8 → убедись, что всё работает (~5 минут).
3. Поставь `SMOKE_TEST = False` и запусти 5–8 снова — полное обучение (несколько часов).
4. Ячейка 9 — оценка на чистом тесте. Ячейка 10 — сохранение 16-битной модели на Hugging Face.

Чекпойнты пишутся на Google Диск: если Colab отключится, перезапусти ячейки 1–8 — обучение продолжится с последнего чекпойнта.

In [ ]:
# Ячейка 1 — GPU и библиотеки
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers peft trl accelerate datasets sacrebleu
import torch, transformers, peft, trl
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
print("bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Ячейка 2 — Google Диск (чекпойнты переживут отключение Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ячейка 3 — вход в Hugging Face (нужен токен с правом записи: huggingface.co/settings/tokens)
# Удобнее всего: слева «ключ» (Secrets) → добавить HF_TOKEN → разрешить доступ ноутбуку.
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Вошёл через Colab Secrets")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# Ячейка 4 — данные: ТОТ ЖЕ prepare_data.py, что на Маке (сплит NLLB v3 + чистый тест)
# Загрузи prepare_data.py из папки проекта на Маке.
import os
if not os.path.exists("prepare_data.py"):
    from google.colab import files
    files.upload()
!python prepare_data.py --train-limit 0 --out data
!wc -l data/*.jsonl

In [ ]:
# Ячейка 5 — настройки
SMOKE_TEST = True          # сначала True (быстрая проверка), потом False (полное обучение)

MODEL_ID  = "google/gemma-4-E4B-it"
RUN_NAME  = "gemma4-e4b-tyv-lora"
DRIVE_DIR = f"/content/drive/MyDrive/tyv-llm/{RUN_NAME}"
HUB_REPO  = "Agisight/tyv-gemma4-e4b"      # приватный репозиторий для слитой 16-битной модели

LORA_R, LORA_ALPHA = 32, 64
LR          = 2e-4
BATCH       = 32           # на A100 40 ГБ; при нехватке памяти — 16
MAX_LEN     = 256
EPOCHS      = 1
SAVE_STEPS  = 1000
EVAL_STEPS  = 1000

if SMOKE_TEST:
    DRIVE_DIR += "-smoke"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Чекпойнты:", DRIVE_DIR)

In [ ]:
# Ячейка 6 — модель и LoRA
import torch
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda")
    print("Загружено через AutoModelForCausalLM")
except Exception as e:
    print("CausalLM не подошёл:", str(e)[:200])
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda")
    print("Загружено через AutoModelForImageTextToText")

model.config.use_cache = False
lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Ячейка 7 — датасет: chat-формат → prompt/completion (loss только по переводу)
from datasets import load_dataset

def to_pc(ex):
    msgs = ex["messages"]
    return {"prompt": msgs[:-1], "completion": msgs[-1:]}

train = load_dataset("json", data_files="data/train.jsonl", split="train").map(to_pc, remove_columns=["messages"])
valid = load_dataset("json", data_files="data/valid.jsonl", split="train").map(to_pc, remove_columns=["messages"])
if SMOKE_TEST:
    train = train.select(range(2000))
valid = valid.select(range(min(500, len(valid))))
print("train:", len(train), "| valid:", len(valid))
print(train[0])

In [ ]:
# Ячейка 8 — обучение (продолжает с последнего чекпойнта на Диске, если он есть)
import glob
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir=DRIVE_DIR,
    num_train_epochs=EPOCHS,
    max_steps=60 if SMOKE_TEST else -1,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    max_length=MAX_LEN,
    completion_only_loss=True,
    logging_steps=10 if SMOKE_TEST else 50,
    eval_strategy="steps",
    eval_steps=30 if SMOKE_TEST else EVAL_STEPS,
    save_strategy="steps",
    save_steps=30 if SMOKE_TEST else SAVE_STEPS,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train, eval_dataset=valid, processing_class=tok)

last = sorted(glob.glob(f"{DRIVE_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
print("Продолжаю с", last[-1] if last else "начала")
trainer.train(resume_from_checkpoint=last[-1] if last else None)
trainer.save_model(f"{DRIVE_DIR}/final_adapter")
print("Адаптер сохранён:", f"{DRIVE_DIR}/final_adapter")

In [ ]:
# Ячейка 9 — оценка chrF++ на ЧИСТОМ тесте (как evaluate.py на Маке)
import json, re, torch
from sacrebleu.metrics import CHRF, BLEU
from prepare_data import PROMPTS

EVAL_LIMIT = 200 if SMOKE_TEST else 100000   # примеров на направление (полный тест = 1369)
tok.padding_side = "left"
model.eval(); model.config.use_cache = True

rows = [json.loads(l) for l in open("data/eval_pairs.jsonl", encoding="utf-8")]
chrf, bleu = CHRF(word_order=2), BLEU()
results = []
for d in ("ru-tyv", "tyv-ru"):
    items = [r for r in rows if r["direction"] == d][:EVAL_LIMIT]
    hyps = []
    for i in range(0, len(items), 64):
        batch = items[i:i+64]
        prompts = [tok.apply_chat_template([{"role": "user", "content": PROMPTS[d].format(src=r["src"])}],
                                           add_generation_prompt=True, tokenize=False) for r in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=256, do_sample=False)
        for o in out[:, enc.input_ids.shape[1]:]:
            h = tok.decode(o, skip_special_tokens=True)
            hyps.append(re.sub(r"<think>.*?</think>", "", h, flags=re.S).strip())
    refs = [r["ref"] for r in items]
    results += [{**r, "hyp": h} for r, h in zip(items, hyps)]
    print(f"{d}: chrF++ = {chrf.corpus_score(hyps, [refs]).score:.1f}  "
          f"BLEU = {bleu.corpus_score(hyps, [refs]).score:.1f}  (n={len(items)})")
    for r, h in list(zip(items, hyps))[:3]:
        print("   src:", r["src"], "\n   hyp:", h, "\n   ref:", r["ref"])

json.dump(results, open(f"{DRIVE_DIR}/eval_results.json", "w"), ensure_ascii=False)
print("Эталон NLLB v3 на чистом тесте: ru→tyv 49.2 | tyv→ru 49.1")

In [ ]:
# Ячейка 10 — слить LoRA с моделью и сохранить в 16 битах (формат для mlx_lm.convert на Маке)
# ВАЖНО: именно 16 бит, не 4-битный bitsandbytes — иначе MLX не прочитает.
assert not SMOKE_TEST, "Сначала полное обучение (SMOKE_TEST = False)"
merged = model.merge_and_unload()
merged.config.use_cache = True
SAVE_DIR = f"{DRIVE_DIR}/merged_bf16"
merged.save_pretrained(SAVE_DIR, safe_serialization=True)
tok.save_pretrained(SAVE_DIR)
merged.push_to_hub(HUB_REPO, private=True)
tok.push_to_hub(HUB_REPO, private=True)
print("Готово:", f"https://huggingface.co/{HUB_REPO}")
print("На Маке:")
print(f"  hf download {HUB_REPO}")
print(f"  mlx_lm.convert --hf-path {HUB_REPO} -q --q-bits 4 --mlx-path models/tyv-gemma4-e4b")